# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a walkthrough for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

The dataset documents clinicopathological and molecular variables for 77 cancer survivors who developed a second primary colorectal cancer (CRC), including demographic, comorbidity, tumor, and treatment history variables, as well as MSI-H (microsatellite instability-high) status and anatomical distribution.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata via Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{getattr(dataset.metadata, 'name', 'Dataset')}: {getattr(dataset.metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

_Note: All entities (record sets, fields) are referenced by their `@id` as per the Croissant standard._

In [ ]:
# List all available record sets and their fields by @id
records_metadata = getattr(dataset.metadata, 'record_sets', [])
if not records_metadata:
    # For some Croissant schemas, try 'record_set' if 'record_sets' is missing
    records_metadata = getattr(dataset.metadata, 'record_set', [])

# Helper to pretty-print record sets and fields
def print_overview(records_metadata):
    print("Available Record Sets and fields (@id):\n")
    all_record_set_ids = []
    if records_metadata:
        for record_set in records_metadata:
            if hasattr(record_set, '@id'):
                rid = record_set['@id'] if isinstance(record_set, dict) else getattr(record_set, '@id')
            else:
                rid = record_set.get('@id', 'unknown') if isinstance(record_set, dict) else 'unknown'
            all_record_set_ids.append(rid)
            name = record_set['name'] if isinstance(record_set, dict) and 'name' in record_set else getattr(record_set, 'name', '(no name)')
            print(f"- RecordSet: {rid} (Name: {name})")
            fields = record_set.get('fields', []) if isinstance(record_set, dict) else getattr(record_set, 'fields', [])
            if not fields:
                # For some schemas might use 'field' instead of 'fields'
                fields = record_set.get('field', []) if isinstance(record_set, dict) else getattr(record_set, 'field', [])
            for field in fields:
                fid = field['@id'] if isinstance(field, dict) and '@id' in field else getattr(field, '@id', 'unknown')
                fname = field['name'] if isinstance(field, dict) and 'name' in field else getattr(field, 'name', '(no name)')
                print(f"    - Field: {fid} (Name: {fname})")
    else:
        print("No record sets found in the metadata.")
    return all_record_set_ids

record_set_ids = print_overview(records_metadata)

# For demonstration, print the first 3 records of each record set using their @id
if record_set_ids:
    for rsid in record_set_ids:
        print(f"\nFirst 3 records from record set '{rsid}':")
        try:
            records = list(dataset.records(record_set=rsid))
            for rec in records[:3]:
                print(rec)
        except Exception as e:
            print(f"Could not load records for {rsid}: {e}")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis. Use record set and field `@id`s obtained above.

In [ ]:
# Extract all data from each record set and store as DataFrames
dataframes = {}
if not record_set_ids:
    print("No record sets found to extract.")
else:
    for rsid in record_set_ids:
        try:
            records = list(dataset.records(record_set=rsid))
            df = pd.DataFrame(records)
            dataframes[rsid] = df
            print(f"Loaded DataFrame for '{rsid}' with shape {df.shape}")
        except Exception as ex:
            print(f"Could not load DataFrame for {rsid}: {ex}")

    # Display columns and a preview for the first record set
    example_set_id = record_set_ids[0]
    print(f"\nColumns for record set '{example_set_id}':")
    print(dataframes[example_set_id].columns.tolist())
    dataframes[example_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We demonstrate typical data cleaning and analysis procedures on one of the main record sets.

This section includes filtering, normalization of a numeric variable, and grouping. All fields are referenced by their `@id`.

**Note:** Change the `numeric_field_id` and `group_field_id` below to match your field IDs from the overview.

In [ ]:
# === EDIT BELOW: Set your actual record set and field @id's ===
# For illustration, we set placeholders based on the likely clinical data
# Replace with values found above if different

# Example record set @id (update if needed):
main_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[main_record_set_id] if main_record_set_id in dataframes else None

if df is not None:
    print(f"Preview of record set '{main_record_set_id}':")
    display(df.head())

    # Try to automatically select a likely numeric field (e.g., age, interval, etc.), or prompt if none.
    numeric_id = None
    for col in df.columns:
        if (df[col].dtype in [int, float]) or (df[col].dtype == object and df[col].str.match(r'^\d+$').all()):
            numeric_id = col
            break
    if not numeric_id:
        print("No numeric field found automatically. Please update 'numeric_field_id'.")
        numeric_field_id = '<your_numeric_field_@id>'
    else:
        numeric_field_id = numeric_id

    # Choose a group field (e.g., sex, tumor_site, msi_status, etc.), as available
    group_field_id = None
    for col in df.columns:
        if any(x in col.lower() for x in ["sex", "gender", "site", "msi", "anatomical"]):
            group_field_id = col
            break
    if not group_field_id:
        group_field_id = df.columns[0]

    # Convert numeric field to numeric if required
    if df[numeric_field_id].dtype == object:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Remove NaN values for demonstration
    filtered_df = df[df[numeric_field_id].notna()]

    # Example filtering: keep only records above mean
    threshold = filtered_df[numeric_field_id].mean()
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df[[numeric_field_id, group_field_id]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nFirst 5 normalized values for '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the group field and compute mean of numeric field
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("No main record set DataFrame available. Please check earlier steps.")

## 5. Visualization
Visualize the distribution of the selected numeric variable and its relationship to a group variable.

In [ ]:
# Basic visualization with matplotlib/seaborn
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    plt.figure(figsize=(12,6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=30)
    plt.show()
else:
    print("Required fields are missing for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² colorectal cancer survivor dataset using the `mlcroissant` library. We:

- Listed available record sets and fields by their Croissant `@id`
- Loaded the main record set into a DataFrame and previewed the data
- Filtered, normalized, and grouped a numeric variable
- Visualized variable distributions and group relationships

This workflow can be adapted to process any dataset described by a Croissant schema. For more detailed analysis or other datasets, adjust the field `@id`s based on your specific schema.